# Notebook F: does a prior supplied in the prompt add to, or interact with, the learned prior behind POPE hallucination? (GPU, about 40 min)

**Background.** On the Gabor task a sentence like "the grating was on the right 8 of the last 10 times" shifted LLaVA's answer toward "right" and shrank the evidence part of the answer: $\Delta_{hist} \approx a\,\Delta_{none} + c\,\mathrm{sign}(\text{count}) + b$ with gain $a\approx0.43$ and offset $c\approx0.63$ logits. POPE hallucination comes from a prior *learned in training* (objects that co-occur are said to be present). Here we ask the same question of POPE: does the in-context yes-count shift $\Delta=\text{logit(Yes)}-\text{logit(No)}$, by how much, and is the shift the same on the three POPE splits (random, popular, adversarial; the learned prior is strongest on adversarial) and on present vs absent objects?

**What is run.** ~510 POPE questions (170 per split, half present / half absent), each with 5 prompts: no history; a neutral sentence of the same length (control); and "In the last 10 questions the answer was yes $k$ times and no $10-k$ times", $k=2,5,8$. One forward pass each, no hooks, no directions (B2 showed that a count-specific direction is not clean enough to steer with).

**What you get.** (1) Baseline POPE behaviour per split. (2) The count effect $\Delta(k{=}8)-\Delta(k{=}2)$ per split and per present/absent, with image-clustered bootstrap CIs. (3) Whether the effect differs between splits (interaction). (4) The effect on the truthful margin (the gain loss) and on the false-yes rate. (5) The gain/offset rule per split, to compare with the gratings.

**Reading the result.** Same shift on every split $\Rightarrow$ the two priors are additive and separate. A larger shift where the learned prior is strong $\Rightarrow$ they interact. A smaller shift there $\Rightarrow$ they compete for the same room. This is behavioural only: it cannot show shared machinery.

**Setup:** GPU on (2x T4), **Internet on** (model and the POPE dataset `lmms-lab/POPE` are downloaded from Hugging Face), Kaggle secret `KAGGLE_KEY` on (for checkpoints), *Save & Run All (Commit)*. If the POPE download fails or has other column names, the loader cell stops with a message that lists the columns it found: send me that message (or the path of your own POPE copy).
**The pipeline was tested on mock data. The Hugging Face loader could not be tested from my environment, so the first cells print a sample and stop with a clear message if something looks wrong.**

In [ ]:
# ===================== CONFIG =====================
MOCK = False
QUICK = False                # True = tiny smoke test (~6 min)
SEED = 0
OUT = "/kaggle/working/F_out"
N_PER_SPLIT = 170                            # questions per POPE split (half present, half absent); 3 splits
KS = (2, 5, 8)                               # in-context 'yes' counts out of 10
NEUTRAL = "The picture was taken with a plain camera in a quiet room on a sunny afternoon."
if QUICK: N_PER_SPLIT = 20
import os, re, glob, json, math, time, zipfile, shutil, warnings, subprocess, sys
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from PIL import Image
warnings.filterwarnings("ignore")
os.makedirs(OUT, exist_ok=True)
SUMMARY = []
def say(s=""): print(s); SUMMARY.append(s)

In [ ]:
# ===================== CHECKPOINTS TO A KAGGLE DATASET (so a dead session loses nothing) =====================
KAGGLE_USER = "ponishbhatia3"
CKPT_SLUG = f"{KAGGLE_USER}/llava-pve-f-ckpt"          # private dataset, created on the first push, new version afterwards
PUSH = not MOCK                                         # set False to disable (then only /kaggle/working is saved)
PUSH_EVERY_MIN = 10
CKPT_DIR = "/kaggle/working/F_ckpt"
RUN_CFG = dict(SEED=SEED, QUICK=QUICK, N_PER_SPLIT=N_PER_SPLIT, KS=list(KS))
json.dump(RUN_CFG, open(f"{OUT}/run_config.json", "w"))
_last_push = 0.0

def _setup_creds():
    assert KAGGLE_USER and "YOUR" not in KAGGLE_USER.upper(), "KAGGLE_USER is a placeholder"
    from kaggle_secrets import UserSecretsClient
    sc, key = UserSecretsClient(), None
    for name in ("KAGGLE_KEY", "KAGGLE_API_TOKEN"):
        try: key = sc.get_secret(name); break
        except Exception: pass
    assert key, "No Kaggle secret found. Add-ons -> Secrets -> add KAGGLE_KEY (your Kaggle API key / token from Settings -> API), switch it ON for this notebook, rerun. Or set PUSH = False."
    os.environ["KAGGLE_USERNAME"] = KAGGLE_USER
    os.environ["KAGGLE_API_TOKEN" if key.startswith("KGAT_") else "KAGGLE_KEY"] = key

def _api():
    from kaggle.api.kaggle_api_extended import KaggleApi
    a = KaggleApi(); a.authenticate(); return a

def push_ckpt(note):
    os.makedirs(CKPT_DIR, exist_ok=True)
    open(f"{CKPT_DIR}/README.txt", "w").write("Checkpoints of notebook F (POPE with an in-context prior). Resume by attaching this dataset as input and rerunning with the same settings.\n")
    for pat in ("*.csv", "*.png", "*.txt", "*.json"):
        for f in glob.glob(f"{OUT}/{pat}"): shutil.copy(f, CKPT_DIR)
    json.dump({"title": "LLaVA PVE F checkpoints", "id": CKPT_SLUG, "licenses": [{"name": "CC0-1.0"}]}, open(f"{CKPT_DIR}/dataset-metadata.json", "w"))
    api = _api()
    try: api.dataset_status(CKPT_SLUG); exists = True
    except Exception: exists = False
    res = api.dataset_create_version(CKPT_DIR, note, quiet=True, dir_mode="zip") if exists else api.dataset_create_new(CKPT_DIR, public=False, quiet=True, dir_mode="zip")
    err = getattr(res, "error", None)
    assert not err, f"Kaggle returned an error: {err}"
    print(f"[checkpoint] {'new version' if exists else 'created dataset'} {CKPT_SLUG} ({note}) | server says: {getattr(res, 'status', res)}", flush=True)

def maybe_push(note, force=False):
    global _last_push
    if not PUSH or (not force and time.time() - _last_push < PUSH_EVERY_MIN * 60): return
    try: push_ckpt(note); _last_push = time.time()
    except Exception as e: print(f"!!! CHECKPOINT PUSH FAILED (the run continues; files are still in {OUT}): {type(e).__name__}: {str(e)[:300]}", flush=True)

# resume: if a previous checkpoint dataset is attached as input with the SAME settings, copy its files back
for hit in glob.glob("/kaggle/input/**/run_config.json", recursive=True):
    try:
        if "llava-pve-outputs-run" in hit or json.load(open(hit)) != json.loads(json.dumps(RUN_CFG)): continue
    except Exception: continue
    d = os.path.dirname(hit)
    for name in ("pope_rows.csv",):
        if os.path.exists(f"{d}/{name}") and not os.path.exists(f"{OUT}/{name}"): shutil.copy(f"{d}/{name}", f"{OUT}/{name}"); print("resumed", name, "from", d)

if PUSH:                                                # fail fast: a real test push now, not after 40 minutes
    _setup_creds(); push_ckpt("preflight")
    print("checkpoint push works; later pushes happen every", PUSH_EVERY_MIN, "min and at the end of each stage")

In [ ]:
# ===================== POPE questions (stratified sample) =====================
class FakeImg:                                   # mock only
    def __init__(s, **kw): s.__dict__.update(kw)
def load_pope():
    if MOCK:
        r = np.random.default_rng(1); rows = []
        for sp in ("random", "popular", "adversarial"):
            for i in range(N_PER_SPLIT):
                rows.append(dict(qid=f"{sp}{i}", split=sp, truth=int(i % 2), question=f"Is there a thing{i % 7} in the image?", img_id=f"img{i % 40}"))
        df = pd.DataFrame(rows); df["image"] = [FakeImg(truth=t, split=s, i=k) for k, (t, s) in enumerate(zip(df.truth, df.split))]; return df
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=False)
    from datasets import load_dataset, concatenate_datasets
    ds = load_dataset("lmms-lab/POPE")
    if hasattr(ds, "keys"): ds = concatenate_datasets([ds[k] for k in ds.keys()])
    cols = ds.column_names; print("POPE columns:", cols, "| rows:", len(ds))
    def pick(*names):
        for n in names:
            if n in cols: return n
        raise AssertionError(f"POPE loader: none of {names} found in columns {cols}. Send me this message.")
    cq, ca, cc, ci = pick("question"), pick("answer", "label"), pick("category", "split", "type"), pick("image")
    cid = next((n for n in ("image_source", "image_id", "image_name") if n in cols), None)
    meta = ds.remove_columns([ci]).to_pandas(); meta["_row"] = np.arange(len(meta))
    meta["truth"] = meta[ca].astype(str).str.lower().str.strip().map({"yes": 1, "no": 0}); assert meta.truth.notna().all(), f"unexpected answers: {meta[ca].unique()[:5]}"
    meta["split"] = meta[cc].astype(str).str.lower(); print("splits found:", meta.split.value_counts().to_dict())
    meta["img_id"] = meta[cid].astype(str) if cid else meta["_row"].astype(str)
    meta["question"] = meta[cq]; meta["qid"] = meta["_row"].astype(str)
    rng = np.random.default_rng(SEED); keep = []
    for sp in ("random", "popular", "adversarial"):
        for t in (1, 0):
            pool = meta[(meta.split == sp) & (meta.truth == t)]._row.values; assert len(pool) >= N_PER_SPLIT // 2, f"split '{sp}' truth {t} has only {len(pool)} questions"
            keep += list(rng.choice(pool, N_PER_SPLIT // 2, replace=False))
    sub = meta[meta._row.isin(keep)].copy(); sub["image"] = [ds[int(r)][ci].convert("RGB") for r in sub._row]; return sub[["qid", "split", "truth", "question", "img_id", "image"]].reset_index(drop=True)
Q = load_pope(); print(Q.groupby(["split", "truth"]).size().unstack()); print("images:", Q.img_id.nunique(), "| example:", Q.question.iloc[0], "| truth", Q.truth.iloc[0])
assert len(Q) == 3 * N_PER_SPLIT

In [ ]:
# ===================== model runner (no hooks needed: only first-token logits) =====================
if MOCK:
    class Runner:
        def prep(s, img, text):
            m = re.search(r"answer was yes (\d+) times", text); k = int(m.group(1)) if m else None; neutral = NEUTRAL in text
            return dict(img=img, k=k, neutral=neutral)
        def delta(s, inp):
            im = inp["img"]; sgn = 2 * im.truth - 1; base = {"random": 3.0, "popular": 2.2, "adversarial": 1.4}[im.split]
            rr = np.random.default_rng(im.i); d = sgn * base + rr.normal(0, 0.4)
            if inp["neutral"]: d += 0.0
            if inp["k"] is not None: d = 0.45 * d + 0.12 * (inp["k"] - 5) * 0.6 + 0.05
            return float(d)
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes", "accelerate"], check=False)
    import torch
    from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig
    NAME = "llava-hf/llava-1.5-7b-hf"
    model = LlavaForConditionalGeneration.from_pretrained(NAME, torch_dtype=torch.float16, device_map="auto", quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)).eval()
    proc = AutoProcessor.from_pretrained(NAME)
    NO_ID, YES_ID = [proc.tokenizer.encode(w, add_special_tokens=False)[0] for w in ("No", "Yes")]
    print("token ids No/Yes:", NO_ID, YES_ID, [proc.tokenizer.convert_ids_to_tokens(i) for i in (NO_ID, YES_ID)])
    class Runner:
        def prep(s, img, text): return proc(images=img, text=text, return_tensors="pt").to(model.device, torch.float16)
        def delta(s, inp):
            with torch.no_grad(): lg = model(**inp).logits[0, -1].float()
            return float((lg[YES_ID] - lg[NO_ID]).item())
RUN = Runner()
def prompt(hist, q):
    core = f"{q}\nAnswer the question using a single word or phrase."
    return f"USER: <image>\n{hist}\n{core} ASSISTANT:" if hist else f"USER: <image>\n{core} ASSISTANT:"
def hist_text(k): return f"In the last 10 questions the answer was yes {k} times and no {10 - k} times."
VARIANTS = ["none", "neutral"] + [f"k{k}" for k in KS]
def vprompt(v, q): return prompt(None, q) if v == "none" else prompt(NEUTRAL, q) if v == "neutral" else prompt(hist_text(int(v[1:])), q)
# self-test: the answer to a clear question should be yes/no with large |delta|
t0 = Q.iloc[0]; d_ = RUN.delta(RUN.prep(t0.image, vprompt("none", t0.question)))
say(f"[self-test] first question: '{t0.question}' truth={t0.truth} split={t0.split}: delta(Yes-No) = {d_:+.2f}")
if not MOCK:
    inp = RUN.prep(t0.image, vprompt("none", t0.question)); top = proc.tokenizer.convert_ids_to_tokens(int(model(**inp).logits[0, -1].argmax())); say(f"[self-test] most likely first token: {top!r} (should be 'Yes' or 'No' with a leading space marker)")
    assert top.replace("▁", "").strip().lower() in ("yes", "no"), "the model does not start its answer with Yes/No: the prompt or token ids are off"

In [ ]:
# ===================== run (checkpointed) =====================
pp = f"{OUT}/pope_rows.csv"; done = pd.read_csv(pp) if os.path.exists(pp) else pd.DataFrame()
have = set(done.qid.astype(str)) if len(done) else set(); rows = done.to_dict("records") if len(done) else []
todo = [i for i in range(len(Q)) if str(Q.qid.iloc[i]) not in have]; t0 = time.time(); print(f"questions: {len(Q)} x {len(VARIANTS)} prompts; left: {len(todo)}")
for n_, i in enumerate(todo):
    r = Q.iloc[i]
    for v in VARIANTS: rows.append(dict(qid=str(r.qid), split=r.split, truth=int(r.truth), img_id=r.img_id, variant=v, delta=RUN.delta(RUN.prep(r.image, vprompt(v, r.question)))))
    if (n_ + 1) % 25 == 0 or n_ == len(todo) - 1:
        pd.DataFrame(rows).to_csv(pp, index=False); maybe_push("pope"); el = time.time() - t0
        print(f"  {n_ + 1}/{len(todo)} | {el / 60:.1f} min | ~{el / (n_ + 1) * (len(todo) - n_ - 1) / 60:.0f} min left", flush=True)
R = pd.DataFrame(rows); R.to_csv(pp, index=False); maybe_push("pope done", force=True)

In [ ]:
# ===================== analysis =====================
from sklearn.metrics import roc_auc_score
W = R.pivot_table(index=["qid", "split", "truth", "img_id"], columns="variant", values="delta").reset_index()
SPL = ["random", "popular", "adversarial"]
say(f"\n[F1] baseline POPE behaviour without history (Delta = logit Yes - logit No), {len(W)} questions")
rows1 = []
for sp in SPL:
    x = W[W.split == sp]; s = 2 * x.truth - 1
    rows1.append(dict(split=sp, n=len(x), AUROC=roc_auc_score(x.truth, x.none), hit_rate=(x[x.truth == 1].none > 0).mean(), false_yes=(x[x.truth == 0].none > 0).mean(), truthful_margin=(s * x.none).mean(), mean_delta_absent=x[x.truth == 0].none.mean()))
say(pd.DataFrame(rows1).round(3).to_string(index=False))
say("(the learned prior should be strongest on 'adversarial': expect the highest false-yes rate and lowest margin there; if all three splits look alike, the test of an interaction has little to work with)")
# ---- image-clustered bootstrap: resample images, keep all their questions in all splits
imgs = W.img_id.values; uimg = np.unique(imgs); idx_by = {u: np.where(imgs == u)[0] for u in uimg}; rb = np.random.default_rng(0)
def boot(stat, n=500):
    out = []
    for _ in range(n):
        pick = rb.choice(uimg, len(uimg)); ii = np.concatenate([idx_by[u] for u in pick]); out.append(stat(W.iloc[ii]))
    return np.array(out)
def ci(draws, j=None):
    d = draws if j is None else draws[:, j]; return np.percentile(d, [2.5, 97.5])
cells = [(sp, t) for sp in SPL for t in (0, 1)]
def gap_stat(x): return np.array([ (x[(x.split == sp) & (x.truth == t)].k8 - x[(x.split == sp) & (x.truth == t)].k2).mean() for sp, t in cells ] + [(x.k8 - x.k2).mean()])
g0, gd = gap_stat(W), boot(gap_stat)
say("\n[F2] count effect = mean of Delta(k=8 of 10 yes) - Delta(k=2 of 10 yes), logits [95% CI, clustered by image]; gratings reference: +0.66")
for j, (sp, t) in enumerate(cells): say(f"  {sp:12s} {'present' if t else 'absent ':8s}: {g0[j]:+.2f} [{ci(gd, j)[0]:+.2f}, {ci(gd, j)[1]:+.2f}]")
say(f"  all questions: {g0[-1]:+.2f} [{ci(gd, len(cells))[0]:+.2f}, {ci(gd, len(cells))[1]:+.2f}]")
say("\n[F3] does the effect differ between splits? (difference of count effects, clustered bootstrap)")
for t, nm in ((0, "absent objects (where hallucination happens)"), (1, "present objects")):
    for sp in ("popular", "adversarial"):
        a, b = cells.index((sp, t)), cells.index(("random", t)); dd = gd[:, a] - gd[:, b]; est = g0[a] - g0[b]
        say(f"  {nm}: {sp} minus random: {est:+.2f} [{np.percentile(dd, 2.5):+.2f}, {np.percentile(dd, 97.5):+.2f}]")
say("\n[F4] evidence cost: swing = mean Delta(present) - mean Delta(absent); ratio to no history (1 = no loss; gratings: about 0.4-0.55 for the count sentence)")
def swing_stat(x):
    o = []
    for sp in SPL:
        y = x[x.split == sp]; sw0 = y[y.truth == 1].none.mean() - y[y.truth == 0].none.mean()
        for v in ("neutral", "k2", "k5", "k8"): o.append((y[y.truth == 1][v].mean() - y[y.truth == 0][v].mean()) / sw0)
    return np.array(o)
s0, sd_ = swing_stat(W), boot(swing_stat); rows4 = []
for a_, sp in enumerate(SPL):
    for b_, v in enumerate(("neutral", "k2", "k5", "k8")): j = a_ * 4 + b_; rows4.append(dict(split=sp, prompt=v, swing_ratio=s0[j], lo=ci(sd_, j)[0], hi=ci(sd_, j)[1]))
say(pd.DataFrame(rows4).round(2).to_string(index=False))
say("\n[F5] hallucination rate: false-yes rate (Delta > 0 on absent objects) and hit rate (present objects)")
rows5 = []
for sp in SPL:
    x = W[W.split == sp]
    for v in ["none", "neutral", "k2", "k5", "k8"]: rows5.append(dict(split=sp, prompt=v, false_yes=(x[x.truth == 0][v] > 0).mean(), hit_rate=(x[x.truth == 1][v] > 0).mean()))
say(pd.DataFrame(rows5).round(3).to_string(index=False))
say("\n[F6] the gratings rule per split: Delta_prompt = a*Delta_none + c*sign + b over the k=2 (sign -1) and k=8 (sign +1) rows; gratings: gain a ~ 0.43, offset c ~ 0.63")
def rule_stat(x):
    o = []
    for sp in SPL + ["all"]:
        y = x if sp == "all" else x[x.split == sp]; dn = np.r_[y.none.values, y.none.values]; sg = np.r_[-np.ones(len(y)), np.ones(len(y))]; yy = np.r_[y.k2.values, y.k8.values]
        b = np.linalg.lstsq(np.c_[dn, sg, np.ones(len(dn))], yy, rcond=None)[0]; o += [b[0], b[1]]
    return np.array(o)
r0, rd = rule_stat(W), boot(rule_stat)
for j, sp in enumerate(SPL + ["all"]): say(f"  {sp:12s} gain a = {r0[2 * j]:.2f} [{ci(rd, 2 * j)[0]:.2f}, {ci(rd, 2 * j)[1]:.2f}] | offset c = {r0[2 * j + 1]:+.2f} [{ci(rd, 2 * j + 1)[0]:+.2f}, {ci(rd, 2 * j + 1)[1]:+.2f}]")
fig, ax = plt.subplots(1, 3, figsize=(16, 4)); w_ = 0.35
for t_, off in ((0, -w_ / 2), (1, w_ / 2)):
    js = [cells.index((sp, t_)) for sp in SPL]; ax[0].bar(np.arange(3) + off, [g0[j] for j in js], w_, yerr=[[g0[j] - ci(gd, j)[0] for j in js], [ci(gd, j)[1] - g0[j] for j in js]], label="present" if t_ else "absent")
ax[0].set_xticks(range(3)); ax[0].set_xticklabels(SPL); ax[0].set_title("count effect on delta (k=8 minus k=2)"); ax[0].legend()
for sp in SPL: ax[1].plot(["none", "k2", "k5", "k8"], [(W[(W.split == sp) & (W.truth == 0)][v] > 0).mean() for v in ["none", "k2", "k5", "k8"]], "o-", label=sp)
ax[1].set_title("false-yes rate on absent objects"); ax[1].legend()
ax[2].bar(range(4), [r0[2 * j] for j in range(4)], yerr=[[r0[2 * j] - ci(rd, 2 * j)[0] for j in range(4)], [ci(rd, 2 * j)[1] - r0[2 * j] for j in range(4)]]); ax[2].axhline(0.43, c="k", ls=":"); ax[2].set_xticks(range(4)); ax[2].set_xticklabels(SPL + ["all"]); ax[2].set_title("gain a (dotted: gratings 0.43)")
plt.tight_layout(); plt.savefig(f"{OUT}/f_overview.png", dpi=130); plt.close()

In [ ]:
open(f"{OUT}/F_summary.txt", "w").write("\n".join(SUMMARY)); maybe_push("final", force=True)
with zipfile.ZipFile(f"{OUT}.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for f in os.listdir(OUT): zf.write(f"{OUT}/{f}", f)
print(f"zip written: {OUT}.zip")